# 03 · 최종 평가와 보고서
F1·F2의 모든 학습과 window 평가가 끝난 뒤에만 실행합니다. 최종 모델을 잠근 다음 보호 데이터(gate·retention)와 RouterArena를 평가하고 보고서를 만듭니다. 결과를 보고 다시 학습하지 않습니다.

## 공통 준비
개인 `alexyoo0506@gmail.com` Colab Pro와 해당 계정의 Drive를 사용합니다. Drive 연결 시 해당 계정을 선택하고 Google 화면에서 직접 승인합니다. GCP·Vertex 제출, GCS 업로드, 운영 `CURRENT` 변경 코드는 없습니다.

제공한 실행 패키지의 `retraining-effect` 폴더를 `MyDrive/LLM_Router/` 아래에 업로드하세요. 입력 데이터와 R0(운영 1.0) 모델은 **R1→R0 전이 실험 때 올린 `MyDrive/LLM_Router/transfer-experiments/inputs/experiment-A`를 읽기 전용으로 재사용**합니다. 경로·SHA가 모두 채워져 있으므로 수정 없이 전체 실행합니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile

BASE = Path("/content/drive/MyDrive/LLM_Router/retraining-effect")
SOURCE_ZIP = BASE / "source" / "retraining-source.zip"
EXPECTED_SOURCE_SHA256 = SOURCE_ZIP.with_suffix(".sha256").read_text().split()[0]
RUN_ID = "retraining-effect-v1-run001"  # F1/F2/비교 모두 같은 ID. 새 계획에는 새 ID
RUN = BASE / "runs" / RUN_ID
LOG = BASE / "logger" / "logger-log.parquet"
# R1->R0 전이 실험 때 올린 입력과 복구된 anchor 모델을 읽기 전용으로 재사용한다.
INPUTS = Path("/content/drive/MyDrive/LLM_Router/transfer-experiments/inputs/experiment-A")
PINS = INPUTS / "source_pins.json"
DATA = INPUTS / "data"

assert len(EXPECTED_SOURCE_SHA256) == 64, "소스 ZIP의 SHA-256 파일이 필요합니다"
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == EXPECTED_SOURCE_SHA256
REPO = Path("/content/router-retraining-source")
with zipfile.ZipFile(SOURCE_ZIP) as z:
    for name in z.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    z.extractall(REPO)
for name, expected in json.loads((REPO / "source-bundle-manifest.json").read_text()).items():
    assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == expected, name
EXPERIMENT = REPO / "ml/notebooks/retraining_effect"
HELPER = EXPERIMENT / "experiment.py"
CONFIG = EXPERIMENT / "experiment_config.json"
ENV = dict(os.environ, PYTHONPATH=str(REPO / "ml/src"))
def run(*args):
    subprocess.run([sys.executable, str(HELPER), *map(str, args)], env=ENV, check=True)
assert PINS.is_file(), f"전이 실험 입력이 필요합니다: {PINS}"
print("Run directory:", RUN)

## 패키지 설치
Colab 메뉴 **런타임 → 런타임 유형 변경 → 런타임 버전 2026.07**(Python 3.12.13), GPU **T4**를 선택합니다. 설치 후 이미 import한 패키지 버전이 다르면 세션을 재시작하고 경로 설정 셀부터 다시 실행하세요.

In [ ]:
assert sys.version_info[:2] in {(3, 11), (3, 12)}, (f"현재 Python {sys.version.split()[0]}. 런타임 버전을 2026.07(Python 3.12)로 바꾸고 처음부터 실행하세요.")
# Text-only model: remove preinstalled optional packages incompatible with pinned torch.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.5.1", "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(EXPERIMENT / "requirements-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO / "ml")], check=True)

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU(T4) 런타임을 선택하세요"
print(torch.cuda.get_device_name(0), torch.__version__, torch.version.cuda)

## 최종 모델 잠금과 보호 데이터 평가
F0(anchor) 1회와 F1/F2 × seed 3개의 최종(3.0) 모델을 평가합니다. T4 약 1시간이 걸립니다. 중단되면 같은 셀을 다시 실행하면 끝난 평가는 검증 후 건너뜁니다.

In [ ]:
if not (RUN / "final-lock.json").exists():
    run("final-lock", "--run-dir", RUN)
for role in ["gate", "forget", "routerarena"]:
    run("score-final", "--run-dir", RUN, "--arm", "f0_frozen", "--seed", 0, "--role", role)
    for arm in ["f1_cycle_only", "f2_replay"]:
        for seed in [42, 43, 44]:
            run("score-final", "--run-dir", RUN, "--arm", arm, "--seed", seed, "--role", role)

## 분석 자료 생성
prequential 곡선, paired 비교, 망각 확인, 계획서 8절 판정을 만듭니다. 누락 결과를 추정치로 채우지 않습니다.

In [ ]:
run("compare", "--run-dir", RUN)
from IPython.display import display, Image, Markdown
import pandas as pd
display(pd.read_csv(RUN / "report/paired-comparisons.csv"))
display(pd.read_csv(RUN / "report/retention-check.csv"))
display(Image(filename=str(RUN / "report/prequential-curves.png")))
display(Markdown((RUN / "report/REPORT.md").read_text()))

## 결과 하나만 전달
아래 셀이 모델 가중치를 제외한 보고서·예측·실행 기록을 `results-for-review.zip` 하나로 내려받습니다. 이 파일만 대화에 첨부하세요.

In [ ]:
review_zip = RUN / "results-for-review.zip"
with zipfile.ZipFile(review_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for name in ["plan.json", "plan.sha256.json", "final-lock.json"]:
        z.write(RUN / name, name)
    z.write(RUN / "shared-data/stream-windows.parquet", "shared-data/stream-windows.parquet")
    for folder in ["report", "scores"]:
        for path in sorted((RUN / folder).rglob("*")):
            if path.is_file():
                z.write(path, path.relative_to(RUN))
    for path in sorted((RUN / "arms").rglob("*.json")):
        relative = path.relative_to(RUN)
        if "checkpoints" not in relative.parts and path.name in {"completed.json", "stage-contract.json", "metrics.json"}:
            z.write(path, relative)
    for path in sorted((RUN / "arms").rglob("attempts/*.json")):
        z.write(path, path.relative_to(RUN))
print("최종 검토용 파일:", review_zip)
from google.colab import files
files.download(str(review_zip))